# Setup

In **Google Colab**, run the setup cell below: it downloads the versioned repository, installs DaliScope, and enables custom widgets. No local Python or Conda installation is needed. Use a standard CPU runtime.

In **local Jupyter**, install DaliScope using the root README and select **Python (DaliScope)**. If you update packages in an existing local session, restart the kernel once.

Run cells in order. The analysis setup locates the repository and sets the working directory to `notebooks` for relative data paths. See the [Colab guide](https://github.com/Liisa-Holm-group/DaliScope/blob/v0.1.4/docs/colab.md) for saving outputs and restarting a cloud runtime.


In [ ]:
# In Colab this installs DaliScope automatically. Locally, follow the README first.
import sys
from pathlib import Path
import subprocess
import runpy

try:
    import google.colab
except ModuleNotFoundError as error:
    if error.name not in {"google", "google.colab"}:
        raise
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    DALISCOPE_RELEASE = "v0.1.4"
    colab_root = Path("/content/DaliScope")
    if not colab_root.exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", DALISCOPE_RELEASE,
            "https://github.com/Liisa-Holm-group/DaliScope.git", str(colab_root),
        ], check=True)
    setup_path = colab_root / "scripts/colab_setup.py"
    if not setup_path.is_file():
        raise RuntimeError("This runtime contains an older checkout. Start a fresh runtime and run setup again.")
    setup = runpy.run_path(str(setup_path))
    setup["setup_colab"](colab_root, DALISCOPE_RELEASE)
else:
    print("Local Jupyter: using the installed DaliScope environment.")


In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd().resolve()
if not (ROOT / "daliscope").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "daliscope").is_dir():
    raise RuntimeError("Start in the DaliScope root or notebooks directory")
os.chdir(ROOT / "notebooks")
import daliscope
print("DaliScope", daliscope.__version__)

In [ ]:
import pandas as pd
from IPython.display import display

import daliscope
import daliscope.core.project
import daliscope.optics.histograms
import daliscope.analysis.data_preview
import daliscope.analysis.occupancy
import daliscope.analysis.structal_similarity # for cluster evaluation
import daliscope.widgets.pairwise_py3Dmol
import daliscope.optics.modules_sunburst
import daliscope.analysis.communities
import daliscope.analysis.pfam_stats

from daliscope.analysis.data_preview import show_heading

# Worked Example: Fish-Eye View of Fold Space Around a GOLD-like Domain with Jelly-Roll Topology

*Using DaliScope functions*

## Background

GOLD domains (Pfam clan CL0521) are small β-sandwich modules — eight strands arranged in two sheets, adopting a jelly-roll topology related to the Nucleoplasmin-like/VP fold — found across a functionally coherent family of proteins involved in Golgi trafficking. We use a representative GOLD-like domain as the query for this worked example: PDB entry 5TDQ, chain A, Golgi resident protein GCP60.

## Search strategy

Starting from this query, we searched AFDB2 using ISS-ProtSci with transitive closure, returning over 12,000 hits — far too many, and far too structurally heterogeneous, to inspect one at a time. As a first step, we modeled the *core* fold as a coherent set of secondary structural elements present across more than 5,000 of these hits, giving us a working definition of "the jelly-roll" independent of any single hit's idiosyncrasies.

## The fish-eye problem

A standard Dali search reports only one thing per hit: a Z-score, the structural similarity of that hit *to the query*. This is a radial measurement — it tells you how far each point sits from the center, but nothing about how the points relate to *each other*. The resulting picture is a fish-eye view: everything is placed relative to a single vantage point, and two hits that are highly similar to one another can appear on opposite sides of the query with no indication that they're related at all. This is a real limitation once a fold turns out — as this one does — to be shared across many distinct superfamilies: radial distance from the query cannot, by construction, distinguish "many independent instances of the same superfamily" from "several structurally distinct superfamilies that each merely resemble the query."

## Approach: the Rigid Core Pipeline

To move beyond this single-vantage-point view, we ran DaliScope's Rigid Core Pipeline: pairwise STRUCTAL rescoring among the hits themselves (not just against the query), followed by Infomap community detection on the resulting similarity network. This identifies communities of mutually similar sub-populations directly from the geometry of the hit set, with no reference to functional annotation. We then use Pfam classification, held out from the clustering itself, to ask whether these purely geometric communities are functionally coherent — and show that they are, with individual communities enriched for specific, related Pfam annotations.

## What this document covers

Working from this community structure, we:

- visualize the overall hit landscape and a Pfam-colored scatterplot of the search results;
- filter to the domain-relevant population and run the STRUCTAL/Infomap community analysis described above;
- summarize each community's characteristic Pfam annotations and free-text description stems.

Taken together, the contrast between functional coherence at the module level and the presence or absence of this signature motif within modules lets us formulate concrete functional hypotheses for hundreds of currently unannotated target proteins. Beyond the specific biological result, this worked example is intended primarily to demonstrate DaliScope's analytical and visualization capabilities end to end — from a raw multi-thousand-hit search result to a structured, testable functional hypothesis.

# Raw Dali data
On loading the data, DaliScope presents a static scatterplot in the (Z-score, query-coverage) plane. These are the two main measures of alignment quality. They are generally correlated. The Dali-score is a sum over similarities of intramolecular distances - a more precise fit gives a higher Z-score with the same number of aligned residues.

The plot uses colour coding for 
- sequence identity regimes
- Z-score regimes

Z-score regimes are based on heuristic thresholds - they usually work for "typical cases". Emprically, Dali Z-scores rank hits in reasonable agreement with expert-curated manual classifications like SCOP. However, the optimal classification thresholds differ from one query structure to another, depending on the shape and size of the protein.

Sequence identity regimes indicate what could be found by traditional sequence search (homologs), where structural alignment improves alignment quality (twilight) and where it is indispensable (dark).

In [ ]:
# Load data
# Colab fetches this optional release asset automatically and checks its SHA-256.
pack_path = 'data/GOLD.tar.gz'
if IN_COLAB:
    subprocess.run([sys.executable, str(ROOT / "scripts/download_example.py"),
                    "GOLD", "--tag", DALISCOPE_RELEASE], check=True)
if not Path(pack_path).is_file():
    raise FileNotFoundError('GOLD pack is optional. See data/README.md for release download instructions.')

# 1. Initialize & Preview Top Hits
project, FULL_DF, FULL_VIEW = daliscope.analysis.data_preview.load_and_inspect_project(pack_path)

# 2. Plot Landscape
fig = daliscope.analysis.data_preview.plot_dali_hits_with_boundaries(
    FULL_DF, 
    daliscope.analysis.data_preview.build_and_register_subsets(project, FULL_DF, FULL_VIEW)
)

>The plot shows a typical mix of sequence identity regimes: homologs have the highest Z-scores, followed by twilight hits. The large peak with dark hits can contain genuine biologically interesting hits, and noise.
>
>DaliScope automatically created two subsets wihch bisect the plane at valleys in the density distribution. Sometimes - but far from always - the subsets correspond to families or superfamilies. These automatically generated subsets are called FILTERED_0 and FILTERED_1.

# Domain-clipped superimposition and occupancy-based selection
Dali alignments optimize intramolecular distance similarity. A global fit can be disrupted by distant aligned residues outside a shared structural core. Here we clip the alignment to a domain and recompute a **uniform least-squares fit** over the retained aligned residues. We then select targets using an occupancy-weighted coverage score. This selection inherits the clipped fit; it does not reweight or refit the coordinates.

The intended structural core is **PDB residues 23–123, inclusive**. The clipping cell converts this to `22-123` in the zero-based, half-open API. Community detection converts the stored clipping provenance back to one-based inclusive ranges, so both steps select exactly the same query residues. The occupancy cutoff is 0.8; it is a weighted score rather than a requirement for 80% unweighted residue coverage.

In [ ]:
# 3. Project structural alignments onto Query positions
project.viz.plot_msa(FULL_VIEW, 0, 123, plot_type='heatmap', data_col='dssp_pileup')

# 2. Preserve the stated PDB structural core in the clipping convention.
from daliscope.mechanics.domain_ranges import pdb_domain_string_to_clipping, clipping_ranges_to_pdb
core_pdb_domains = "23-123"
daliscope.mechanics.metrics.register_multiple_domains(
    domain_string=pdb_domain_string_to_clipping(core_pdb_domains), project=project, custom_name="dom",
)

# 3. Register filtered FOLD subset. 
daliscope.analysis.occupancy.create_fold_subset(project, 'dom', show_plot=False, cutoff=0.8)

# 4. Compare rmsd in raw Dali data, after domain clipping, and after FOLD ocuupancy thresholding
df = FULL_DF[['target_id','rmsd']].copy()
df['clipped & filtered'] = project.views['dom_FOLD']['rmsd']
df['clipped'] = project.views['dom']['rmsd']
daliscope.optics.histograms.plot_overlaid_distributions(
    [df['rmsd'], df['clipped'], df['clipped & filtered']], 
    figsize=(6,2),
    title='RMSD distribution'
)

# Play with the structures
The Query structure is blue, the target is red. Thick lines and dashed lines show the raw Dali alignment, but the least-squares superimposition uses the clipped aligned residues with uniform weights.

In [ ]:
daliscope.widgets.pairwise_py3Dmol.generate_superimposition_view(
    target_ix=2777,
    df=project.views['dom_FOLD'],
    project=project,
    q_ca_coords=project.query_ca_coords,
    target_ca_coords=project.coords,
)

# What do existing domain classifications tell us?
Here, we show the Pfam family and superfamily (clan) assignments of the domain-fold populations.

The table links family and clan identifiers to InterPro documentation. The clan descriptions confirm that these populations include jellyroll folds, a common structural topology.

A notable fraction of the targets lack Pfam assignments despite structural similarity to the domain fold, as shown by the current “Unassigned” count below. Such gaps in domain classification occur even among targets with high Z-scores. In the domain cartoons, these targets appear as empty dashed red boxes; in the scatterplot, they appear as green dots dispersed among known GOLD-like domains (CL0521).

These gaps in domain classification motivate a complementary question: **what structural communities exist within the domain-fold population, independent of sequence or existing annotation?**

In [ ]:
# 1. Table view
daliscope.analysis.pfam_stats.pfam_counts('dom_FOLD', project, k=10, print_tables = True)

In [ ]:
# 2. Domain architecture cartoons
project.viz.architecture('dom_FOLD', renderer='matplotlib', pfam_col='pfam_domains')

In [ ]:
# 3. Scatterplot colored by Pfam categories
df = project.views['dom_FOLD'][['alignment_length', 'z_score', 'clan', 'pfam']].copy()

#df["pfam_label"] = daliscope.analysis.pfam_stats.top_category_label(df, "pfam", k=10)
df["clan_label"] = daliscope.analysis.pfam_stats.top_category_label(df, "clan", k=10)

daliscope.optics.histograms.plot_scatter(df, x='z_score', y='alignment_length', color='clan_label', engine='matplot',)

# Community detection

In [ ]:
# Run Rigid Core Pipeline
# Use the same physical query residues as the clipped view.
community_ranges = clipping_ranges_to_pdb(project.provenance["dom"].parameters["domain_ranges"])

# Call community detection function across dom_FOLD
modules_df, n_modules = daliscope.analysis.communities.run_community_detection(
    view_name="dom_FOLD",
    domain_range=community_ranges,
    project=project,
    k_neighbors=200,
    max_workers=1,
    verbose=True,
)

# Sunburst diagrams. Colors are recycled after 18 clans
fig, axes, clan_colors = daliscope.optics.modules_sunburst.plot_module_sunbursts(
    modules_df,
    modules=range(1, n_modules+1),
)

# Cross-tabulating pfam catefories and modules
daliscope.analysis.communities.module_pfam_table(project, modules_df, row_group='clan', min_count=20)
daliscope.analysis.communities.module_pfam_table(project, modules_df, row_group='pfam', min_count=20)

# Module IDs are run-specific. Choose representatives by current annotation counts.
def representative_module(column, accession):
    counts = modules_df.loc[modules_df[column] == accession].groupby("module").size()
    if counts.empty:
        return None
    return int(counts.sort_values(ascending=False, kind="stable").index[0])


def recurse_annotation_module(column, accession):
    module_id = representative_module(column, accession)
    if module_id is None:
        print(f"No current module has {column}={accession}; skipping recursion.")
        return
    selected_view = f"dom_FOLD_module_{module_id}"
    if len(project.views[selected_view]) < 2:
        print(f"{selected_view} has fewer than two targets; skipping recursion.")
        return
    show_heading(f"{selected_view}: most targets annotated {column}={accession}")
    child_modules, n_child_modules = daliscope.analysis.communities.run_community_detection(
        view_name=selected_view, domain_range=community_ranges, project=project,
        k_neighbors=200, max_workers=1, verbose=True,
    )
    daliscope.optics.modules_sunburst.plot_module_sunbursts(
        child_modules, modules=range(1, n_child_modules + 1), figsize=(8, 2),
    )
    display(child_modules[["module", "pfam", "clan"]].value_counts().rename("count").reset_index())


## Recursive splitting
Inspect whether selected modules remain cohesive or split into finer structural populations. Module numbers are assigned by the current Infomap run and are not fixed biological labels.

The following cells select the module with the most current **CL0100 (C1q_TNF)** annotations and the module with the most **PF17065** annotations. Ties use the lowest module ID. They display the new submodule annotations; missing annotations or fewer than two targets are explicitly skipped. Selection by annotation guides inspection and does not establish a shared function for every member.

In [ ]:
recurse_annotation_module("clan", "CL0100")

In [ ]:
recurse_annotation_module("pfam", "PF17065")

Other registered module views can be inspected by selecting their current annotations or view names. Reuse `community_ranges` when running additional community analyses.

## How structurally distinct are the modules?
To evaluate module independence across the whole population, we compare the within- and between-module distributions of STRUCTAL scores. While this population-level metric confirms that within-module alignments are significantly elevated over background, global score comparisons carry the caveat that class imbalance can obscure internal community substructure—an aspect directly resolved by the recursive approach above.

In [ ]:
# Statistical evaluation of pairwise STRUCTAL scores between and within modules
_summary_df, raw_scores = daliscope.analysis.structal_similarity.evaluate_module_cohesion(
    project, 'dom_FOLD', modules_df,
    n_within=2000, n_between=8000,
    modules=None,  # or None for everything, with the cost warning
)

# Box-plot
fig, ax = daliscope.analysis.communities.plot_module_score_distributions(
    raw_scores=raw_scores,
    figsize=(12, 6),
    title="Within- and between-module STRUCTAL scores (2000 + 8000 randomly sampled)",
)

# What structural and functional features characterize a population?
We demonstrate structural superimposition, domain architecture cartoons, and summaries of short functional descriptions.

## Transitive superimposition
Select the current module containing the most **PF00190 (Cupin_1)** annotations, rather than relying on a historical module number. PF00190 includes diverse cupin proteins; inspect target descriptions and structures before assigning a specific function such as germin activity.

The cell displays up to five evenly spaced targets in that module. The query is a GOLD-like domain, shown as a thin gray reference trace. Missing annotations are explicitly skipped.

Mouse controls: rotate, zoom, and pan.

In [ ]:
import numpy as np

module_id = representative_module("pfam", "PF00190")
if module_id is None:
    print("No current PF00190 module; skipping transitive superimposition.")
else:
    view_name = f"dom_FOLD_module_{module_id}"
    selected_df = project.views[view_name]
    target_ixes = np.linspace(0, len(selected_df) - 1, min(5, len(selected_df)), dtype=int).tolist()
    show_heading(f"{view_name}: representative PF00190 annotation population")
    view = daliscope.widgets.pairwise_py3Dmol.generate_transitive_superimposition_view(
        target_ixes=target_ixes, df=selected_df, project=project,
        target_ca_coords=project.coords, q_ca_coords=project.query_ca_coords,
        query_radius=0.15, show_query=True,
    )
    view.show()

## Summarizing functional descriptions per module (t.b.i.)
For each module, we select representative functional descriptions using a TF-IDF-based approach.

This section is a planned extension and is not executed in this notebook.

# Summary of Workflow

In this worked example, we demonstrated how **DaliScope** moves beyond basic homology searching:

1. **Beyond Top Hits:** We explored deep structural alignments rather than relying solely on the single highest-scoring match.
2. **User-Driven Decisions:** You retained full control throughout the workflow—from clipping domains and defining active/binding sites to inspecting the frozen Pfam annotations and their InterPro links.
3. **Reproducible Output:** Because this run relies on a self-contained, frozen data pack, re-running these notebook cells will produce reproducible analysis with the recorded software environment and parameters.